### Environment Setup and Library Imports

Before building our logistic regression model, we need to set up our environment by importing the necessary libraries and downloading required linguistic resources.

* **`pandas`**: Used for structuring our data into easy-to-use DataFrames.
* **`datasets`**: A library by Hugging Face that allows us to easily load and manage text classification datasets.
* **`nltk`, `re`, `string`**: These are essential tools for text preprocessing. `re` handles regular expressions for cleaning text, `string` provides sets of common punctuation, and `nltk` (Natural Language Toolkit) gives us powerful text processing capabilities.
* **`nltk.download('punkt')`**: This downloads the 'punkt' tokenizer model, which NLTK needs to accurately split our text into individual words or sentences later on.


In [1]:
# !pip install datasets nltk
import pandas as pd
from datasets import load_dataset
import nltk
import re
import string

# Download necessary NLTK data (if using for tokenization)
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

### Loading and Inspecting the Dataset

With our environment set up, the next step is to load the data we will be classifying. 

* **`load_dataset("sh0416/ag_news")`**: We are fetching the AG News dataset directly from the Hugging Face hub. This is a popular benchmark dataset consisting of news articles categorized into four distinct classes (World, Sports, Business, and Sci/Tech).
* **Inspecting the Structure**: Printing the `dataset` object shows us its splits (typically `train` and `test` sets). 
* **Converting to Pandas**: While Hugging Face datasets are highly efficient, converting the training split into a pandas `DataFrame` (`train_df`) allows us to easily visualize the data in a tabular format and inspect the first few rows to understand the features (like the text content and their corresponding integer labels).


In [2]:
# Loading the dataset from Hugging Face
dataset = load_dataset("sh0416/ag_news")

# Displaying the structure of the dataset
print(dataset)

# Convert to a Pandas DataFrame for easier viewing
train_df = pd.DataFrame(dataset['train'])
print(f"\nFirst few rows of training data:\n{train_df.head()}")

DatasetDict({
    train: Dataset({
        features: ['label', 'title', 'description'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['label', 'title', 'description'],
        num_rows: 7600
    })
})

First few rows of training data:
   label                                              title  \
0      3  Wall St. Bears Claw Back Into the Black (Reuters)   
1      3  Carlyle Looks Toward Commercial Aerospace (Reu...   
2      3    Oil and Economy Cloud Stocks' Outlook (Reuters)   
3      3  Iraq Halts Oil Exports from Main Southern Pipe...   
4      3  Oil prices soar to all-time record, posing new...   

                                         description  
0  Reuters - Short-sellers, Wall Street's dwindli...  
1  Reuters - Private investment firm Carlyle Grou...  
2  Reuters - Soaring crude prices plus worries\ab...  
3  Reuters - Authorities have halted oil export\f...  
4  AFP - Tearaway world oil prices, toppling reco...  


### NLTK Setup and Tokenization Test

Here, we are ensuring that our text tokenization tools are fully configured and ready to use, particularly making sure they are compatible with newer Python environments.

* **Downloading Tokenizer Data**: We use `nltk.download('punkt')` and `nltk.download('punkt_tab')`. The `punkt` package contains pre-trained models that NLTK uses to divide text into lists of sentences or words. The `punkt_tab` addition is specifically required to prevent errors in Python 3.11 and newer versions.
* **Testing `word_tokenize`**: Before applying tokenization to our entire dataset, it's good practice to verify that the tool works. We import `word_tokenize` and run a quick test on a sample sentence (`"Hello world, it works!"`) to confirm it successfully splits the string into individual words and punctuation marks.

In [3]:
import nltk

# Download the specific table needed for Python 3.11+ environments
nltk.download('punkt')
nltk.download('punkt_tab')

# Verify it works
from nltk.tokenize import word_tokenize
print("Test Tokenization:", word_tokenize("Hello world, it works!"))

Test Tokenization: ['Hello', 'world', ',', 'it', 'works', '!']


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\LENOVO\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


### Text Preprocessing Pipeline

This cell defines the core function we will use to clean and standardize our raw text data. Raw text contains a lot of noise, and applying these steps ensures our logistic regression model can focus on the actual vocabulary to learn patterns effectively.

Here is what the `preprocess_text` function does step-by-step:
* **Lowercasing**: Converts all text to lowercase. This ensures uniformity so that the model treats words like "Success" and "success" identically, reducing the overall size of our vocabulary.
* **Removing Punctuation**: Uses Python's `string.punctuation` and translation tables to strip out grammatical marks (like commas, periods, and exclamation points). In basic topic classification, punctuation usually doesn't add predictive value.
* **Tokenization**: Uses the `word_tokenize` function we configured earlier to split the cleaned, continuous string into a list of distinct, individual words (tokens). 

Finally, we run a quick test on a sample string to verify the output. The result is a clean list of normalized tokens, perfectly prepped for the next stage of our NLP pipeline.

In [4]:
import string
from nltk.tokenize import word_tokenize

def preprocess_text(text):
    # Normalization: Lowercase
    text = text.lower()
    
    # Normalization: Remove Punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Tokenization
    # Now that 'punkt_tab' is downloaded, this will run smoothly
    tokens = word_tokenize(text)
    
    return tokens

# Test
sample = "Success! The tokenizer is now working."
print(preprocess_text(sample))

['success', 'the', 'tokenizer', 'is', 'now', 'working']


In [5]:
print(dataset['train'].column_names)

['label', 'title', 'description']


### Applying Preprocessing to the Entire Dataset

Now that we have a working preprocessing function, we need to apply it to all the news articles in our dataset. 

* **`dataset.map(...)`**: Hugging Face's `map` function is a highly optimized tool for applying transformations across entire datasets quickly and efficiently. 
* **Lambda Function**: We use a short `lambda` function to take each row (`x`), grab the raw text from the `description` column, and pass it through our `preprocess_text` function.
* **Creating a New Feature**: By returning a dictionary `{"tokenized_text": ...}`, we instruct the dataset to store the output in a newly created column named `tokenized_text`. This keeps our original raw text intact while adding the cleaned, tokenized list of words right alongside it.
* **Verification**: Finally, we print the first item in the training set to confirm that the pipeline worked and the new column contains the expected list of cleaned tokens.

In [6]:

processed_dataset = dataset.map(lambda x: {"tokenized_text": preprocess_text(x['description'])})

# Check a sample to verify
print(f"Tokenized sample: {processed_dataset['train'][0]['tokenized_text']}")

Tokenized sample: ['reuters', 'shortsellers', 'wall', 'streets', 'dwindlingband', 'of', 'ultracynics', 'are', 'seeing', 'green', 'again']


### Extracting Bigrams and Building a Vocabulary

In this step, we are moving beyond single words (unigrams) to capture slightly more context from our text by creating **bi-grams** (pairs of consecutive words). This is crucial for models like Logistic Regression because it helps the model understand that a phrase like "not good" has a completely different meaning than just the words "not" and "good" separately.



Here is a detailed breakdown of what this code is doing:

* **The `get_bigrams` Function**: We use a very efficient Python trick here: `zip(tokens, tokens[1:])`. By zipping the original list of tokens with a version of itself shifted by one position, we instantly generate consecutive pairs. For example, `["the", "quick", "brown"]` becomes `[("the", "quick"), ("quick", "brown")]`.
* **Aggregating All Bigrams**: We iterate through our entire training dataset, extract the bigrams for each article, and dump them all into one massive list called `all_bigrams`. 
* **Creating the Vocabulary Mapping (`bigram_to_idx`)**: 
    * First, we convert the massive list into a `set` to filter out all duplicates, leaving only the *unique* bigrams. 
    * We then `sort` them to ensure our vocabulary is always in the exact same alphabetical order every time we run the code.
    * Finally, we use a dictionary comprehension to assign a unique integer index to each bi-gram (e.g., `{"(apple, pie)": 0, "(bad, news)": 1}`). This dictionary acts as our lookup table when we convert our text to numbers.
* **Memory Management (`gc.collect`)**: Text processing can eat up a lot of RAM. The `all_bigrams` list we created likely contained millions of items. Since we only needed it temporarily to find the unique items, we use `del all_bigrams` and Python's Garbage Collector (`gc.collect()`) to immediately free up that memory and prevent the notebook from crashing.

In [7]:
from collections import Counter
import numpy as np

def get_bigrams(tokens):
    # Generates a list of tuples: (word1, word2)
    return list(zip(tokens, tokens[1:]))

# Extract all bi-grams from the training set to build the vocabulary
all_bigrams = []
for tokens in processed_dataset['train']['tokenized_text']:
    all_bigrams.extend(get_bigrams(tokens))

#  Get unique bi-grams and create a mapping (index to bi-gram)
# Using a set for unique items, then sorting for consistency
unique_bigrams = sorted(list(set(all_bigrams)))
bigram_to_idx = {bg: i for i, bg in enumerate(unique_bigrams)}

print(f"Total unique bi-grams (Vocabulary Size): {len(unique_bigrams)}")

# Clean up memory as per your "Additional Notes"
import gc
del all_bigrams
gc.collect()

Total unique bi-grams (Vocabulary Size): 960376


0

### Vectorizing the Text: Binary Bigram Encoding



Logistic Regression models cannot read raw text; they require numerical input. This cell bridges that gap by converting each article into a mathematical vector—specifically, a binary "Bag-of-Bigrams" representation.

Here is how the `create_binary_vector` function operates:
* **Initialization**: We create an array of zeros (`np.zeros`) exactly as long as our total vocabulary. We intentionally use `np.float32` instead of the default `float64` to halve our memory consumption—a critical optimization when working with large vocabularies.
* **Feature Extraction**: We extract the bigrams from the current document using our `get_bigrams` function.
* **Binary Encoding**: We iterate through the document's bigrams. If a bigram exists in our master vocabulary (`mapping`), we find its specific integer index and flip the `0` to a `1.0` at that position. 

> **Note:** This is a *binary* encoding. We only care if the bigram is present (1) or absent (0) in the text, ignoring how many times it actually appears. 

Finally, we run a test on a sample sentence to ensure the vector length correctly matches our massive vocabulary size and that the active indices are successfully marked as `1.0`.

In [8]:
def create_binary_vector(tokens, vocab_size, mapping):
    # Initialize a vector of zeros using float32 (prefer over float64 for memory)
    vec = np.zeros(vocab_size, dtype=np.float32)
    
    # Get bi-grams for the current text
    text_bigrams = get_bigrams(tokens)
    
    # Set index to 1 if the bi-gram exists in the vocabulary
    for bg in text_bigrams:
        if bg in mapping:
            vec[mapping[bg]] = 1.0
            
    return vec

# Verification with the example from your assignment image:
# "i love this movie very much"
sample_tokens = ["i", "love", "this", "movie", "very", "much"]
test_vec = create_binary_vector(sample_tokens, len(unique_bigrams), bigram_to_idx)

print(f"Vector length: {len(test_vec)}")
print(f"Indices marked as 1: {np.where(test_vec == 1.0)[0]}")

Vector length: 960376
Indices marked as 1: [418125 851291 905712]


In [9]:
# Shuffle the dataset (important for training quality)
train_data = processed_dataset['train'].shuffle(seed=42)

# Vectorized operation: 
# Since a full dense matrix for 120,000 samples x ~100,000 bi-grams 
# will definitely exceed 12GB, we will process in batches during training.
print("Data shuffled and ready for vectorization.")

Data shuffled and ready for vectorization.


### Implementing Softmax Regression from Scratch





### 1. Initialization (`__init__`)
We set up our hyperparameters: `learning_rate` ($\alpha$), the number of iterations, and `n_classes` (which is 4 for AG News). Crucially, the shape of our `weights` matrix $W$ will be `(n_features, n_classes)` and our `bias` vector $b$ will be `(n_classes,)`. This means the model learns a distinct set of weights for *every single class*.

### 2. The Softmax Function (`_softmax`)
This function converts the raw linear scores (often called *logits*) into normalized probabilities that sum to 1. 

The standard mathematical equation for the Softmax function for a given class $j$ out of $K$ classes is:
$$P(y=j \mid \mathbf{x}) = \frac{e^{z_j}}{\sum_{k=1}^{K} e^{z_k}}$$
*(where $z$ represents our raw scores).*

**The Numerical Stability Trick:** If a raw score $z$ is very high, computing $e^z$ will result in an overflow error (the number gets too big for the computer's memory). To fix this, you implemented a clever mathematical trick: subtracting the maximum score of the row from all scores in that row before exponentiation. Because of exponent rules, this shifts the highest value to $e^0 = 1$, preventing overflow without changing the final probability distribution at all:
$$\frac{e^{z_j - \max(\mathbf{z})}}{\sum_{k=1}^{K} e^{z_k - \max(\mathbf{z})}}$$

### 3. The Training Step (`fit_batch`)
This method represents a single iteration of **Gradient Descent** over a batch of data. 

* **Step 1: Forward Pass**
  First, we compute the linear combination of our inputs and weights, then apply Softmax to get our predicted probabilities ($A$):
  $$Z = XW + b$$
  $$A = \text{softmax}(Z)$$
* **Step 2: One-Hot Encoding**
  Our true labels (`y_true`) are just integers (e.g., class `2`). To compute the error against our 4-column probability matrix, we convert these integers into binary vectors (e.g., class 2 becomes `[0, 0, 1, 0]`). We'll call this true label matrix $Y$.
* **Step 3: Computing Gradients (Backpropagation)**
  This is the heart of the algorithm. We need to know how much to adjust our weights to reduce the Cross-Entropy loss. 
  First, we find the error matrix ($E$) by subtracting the true labels from our predicted probabilities:
  $$E = A - Y$$
  Then, using calculus (the chain rule applied to the loss function), we calculate the gradients ($dW$ and $db$). We divide by $m$ (the number of samples, `n_samples`) to get the *average* gradient:
  $$dW = \frac{1}{m} X^T E$$
  $$db = \frac{1}{m} \sum_{i=1}^{m} E^{(i)}$$
* **Step 4: Update Parameters**
  Finally, we nudge our weights and biases in the opposite direction of the gradient by a step size determined by the learning rate ($\alpha$):
  $$W = W - \alpha dW$$
  $$b = b - \alpha db$$

### 4. Making Predictions (`predict`)
To predict a new label, we run the forward pass to get the probabilities. Then, `np.argmax(probs, axis=1)` simply looks at the probability output for the 4 classes and returns the index of the highest one (e.g., if the probabilities are `[0.1, 0.8, 0.05, 0.05]`, it returns class `1`).

In [ ]:
import numpy as np

class SoftmaxRegressionScratch:
    def __init__(self, learning_rate=0.01, n_iters=1000, n_classes=4):
        self.lr = learning_rate
        self.n_iters = n_iters
        self.n_classes = n_classes
        self.weights = None 
        self.bias = None    

    def _softmax(self, z):
        # Subtracting max(z) for numerical stability
        exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    def fit_batch(self, X, y_true):
        n_samples, n_features = X.shape
        
        if self.weights is None:
            # Explicitly using float32 as required
            self.weights = np.zeros((n_features, self.n_classes), dtype=np.float32)
            self.bias = np.zeros(self.n_classes, dtype=np.float32)

        # Forward Pass: Use X.dot() for sparse compatibility
        scores = X.dot(self.weights) + self.bias
        probs = self._softmax(scores)

        # One-Hot Encoding
        y_one_hot = np.zeros((n_samples, self.n_classes), dtype=np.float32)
        y_one_hot[np.arange(n_samples), y_true] = 1

        # Compute Gradients
        error = probs - y_one_hot
        # X.T.dot(error) handles sparse matrix transposition math efficiently
        dw = (1 / n_samples) * X.T.dot(error)
        db = (1 / n_samples) * np.sum(error, axis=0)

        # Update Parameters
        # Ensure updates remain float32
        self.weights -= self.lr * dw.astype(np.float32)
        self.bias -= self.lr * db.astype(np.float32)

    def predict(self, X):
        # Use X.dot() here as well to handle sparse input
        scores = X.dot(self.weights) + self.bias
        probs = self._softmax(scores)
        return np.argmax(probs, axis=1)

In [ ]:
from scipy import sparse
import numpy as np

def create_sparse_binary_vector(tokens, vocab_size, mapping):
    # Get bi-grams from the current tokens
    text_bigrams = get_bigrams(tokens)
    
    # Map bi-grams to their unique indices in the vocabulary
    indices = [mapping[bg] for bg in text_bigrams if bg in mapping]
    
    # Remove duplicates to ensure "binary" representation (1 if exists)
    unique_indices = list(set(indices))
    
 
    # data: the values (all 1.0 for binary)
    # rows: all zeros because it's a single row vector
    # cols: the vocabulary indices where the bi-gram exists
    data = np.ones(len(unique_indices), dtype=np.float32)
    rows = np.zeros(len(unique_indices), dtype=np.int32)
    cols = np.array(unique_indices, dtype=np.int32)
    
    #  Build the Compressed Sparse Row (CSR) matrix
    return sparse.csr_matrix((data, (rows, cols)), shape=(1, vocab_size))

### The Training Loop: Mini-Batch Gradient Descent




### 1. Hyperparameters & Initialization
You define the rules of engagement before training begins:
* **`learning_rate` ($\alpha$)**: Set to **0.1**, this controls the step size during gradient descent.
* **`epochs`**: Set to **5**, meaning the model will see the entire training dataset exactly five times.
* **`batch_size`**: Set to **64**. The model will look at 64 articles at a time, calculate the average error, and update its weights. 

Mathematically, if you have $N$ training samples and a batch size of $B$, the model performs $\lceil \frac{N}{B} \rceil$ weight updates per epoch.

### 2. Shuffling the Data
```python
np.random.shuffle(indices)

In [13]:
from scipy import sparse
import gc

# Hyperparameters
learning_rate = 0.1
epochs = 5
batch_size = 64
n_classes = 4 

# Initialize the sparse-compatible model
model = SoftmaxRegressionScratch(learning_rate=learning_rate, n_classes=n_classes)

for epoch in range(epochs):
    # Shuffle training data each epoch
    indices = np.arange(len(train_data))
    np.random.shuffle(indices)
    
    epoch_correct = 0
    total_samples = 0
    
    
    for i in range(0, len(indices), batch_size):
        batch_indices = indices[i:i + batch_size]
        
        # Vectorize on the fly to save RAM
        batch_tokens = [train_data['tokenized_text'][idx] for idx in batch_indices]
        
        # Use the sparse version of your vectorizer
        X_batch_list = [create_sparse_binary_vector(t, len(unique_bigrams), bigram_to_idx) 
                        for t in batch_tokens]
        
        # Efficiently stack the sparse vectors
        X_batch = sparse.vstack(X_batch_list)
        
        # Shift labels (1,2,3,4 -> 0,1,2,3)
        y_batch = np.array([train_data['label'][idx] - 1 for idx in batch_indices], dtype=np.int32)
        
        # Train on the mini-batch
        model.fit_batch(X_batch, y_batch)
        
        # Calculate training accuracy for feedback
        predictions = model.predict(X_batch)
        epoch_correct += np.sum(predictions == y_batch)
        total_samples += len(y_batch)
        
        # Free unused memory using del
        del X_batch, X_batch_list, y_batch
        
    # Calculate and print accuracy for the whole epoch
    accuracy = (epoch_correct / total_samples) * 100
    print(f"Epoch {epoch+1}/{epochs} completed. Train Accuracy: {accuracy:.2f}%")
    
    # Manual garbage collection for 12GB limit
    gc.collect()

Epoch 1/5 completed. Train Accuracy: 74.01%
Epoch 2/5 completed. Train Accuracy: 80.39%
Epoch 3/5 completed. Train Accuracy: 83.45%
Epoch 4/5 completed. Train Accuracy: 85.23%
Epoch 5/5 completed. Train Accuracy: 86.65%


### Benchmarking Against Scikit-Learn: The Final Comparison




### 1. Replicating Your Custom Vectorization
To make the comparison fair, Scikit-Learn must see the exact same mathematical representation of the text that your custom model saw. 
Instead of writing a manual loop, we use `TfidfVectorizer`, but we intentionally "turn off" its advanced features to mimic your binary bi-gram function:
* **`ngram_range=(2, 2)`**: Forces the extractor to *only* look at bi-grams (ignoring single words), matching your `get_bigrams` logic.
* **`binary=True`**: Marks a `1` if the bi-gram is present and `0` if it is not, ignoring the actual frequency count.
* **`use_idf=False` & `norm=None`**: Disables Inverse Document Frequency scaling and length normalization, ensuring the output is a raw binary matrix, just like your `create_binary_vector` output.

### 2. Model 1: Scikit-Learn `LogisticRegression`
This is the standard implementation. Under the hood, it does not use the simple Gradient Descent loop you wrote. Instead, it uses highly optimized, C-compiled mathematical solvers (like `liblinear` or `lbfgs`).
* **Why it might perform differently:** Scikit-Learn applies **L2 Regularization** by default. This mathematically shrinks the weights to prevent overfitting, which often results in a slightly higher test accuracy compared to an unregularized from-scratch model.

### 3. Model 2: Scikit-Learn `SGDClassifier`
This model is actually the closest equivalent to what you built! 
* **`loss='log_loss'`**: By setting the loss function to Log Loss (Cross-Entropy), you instruct the SGD Classifier to behave exactly like a Logistic/Softmax Regression model. 
* The mathematical objective here is to minimize the Cross-Entropy Loss:
  $$L(y, p) = -\sum_{c=1}^{K} y_{o,c} \log(p_{o,c})$$
  *(where $y$ is the true binary indicator and $p$ is the predicted probability for class $c$).*
* **Optimization:** SGD stands for **Stochastic Gradient Descent**. Unlike your mini-batch approach (updating weights every 64 samples), pure SGD calculates the gradient and updates the weights after *every single sample*. It is incredibly fast and memory-efficient for massive text datasets.

### 4. The Verdict
Finally, the code prints a side-by-side comparison of the accuracies. Because Scikit-Learn models are written in highly optimized C/C++ and Cython, and include built-in regularization, they generally train faster and might score a few percentage points higher. However, seeing your scratch model achieve a comparable accuracy proves that the underlying math you implemented is perfectly sound!

*(Note: We conclude the cell with standard memory cleanup using `del` and `gc.collect()` to free up the RAM consumed by the massive Scikit-Learn sparse matrices).*

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score
import gc

# Prepare text data for Sklearn 
train_texts = dataset['train']['description']
test_texts = dataset['test']['description']
y_train_sk = np.array(dataset['train']['label']) - 1 # Shift labels to 0-3
y_test_sk = np.array(dataset['test']['label']) - 1

# Feature Extraction (Matching your bi-gram logic)
# We use binary=True to match your "1 if occurs, else 0" 
vectorizer = TfidfVectorizer(ngram_range=(2, 2), binary=True, use_idf=False, norm=None, dtype=np.float32)

print("Vectorizing data for Scikit-learn...")
X_train_sk = vectorizer.fit_transform(train_texts)
X_test_sk = vectorizer.transform(test_texts)

# sklearn.linear_model.LogisticRegression
print("Training Sklearn LogisticRegression...")
lr_sk = LogisticRegression(max_iter=100, solver='liblinear') 
lr_sk.fit(X_train_sk, y_train_sk)
lr_acc = accuracy_score(y_test_sk, lr_sk.predict(X_test_sk))

#  sklearn.linear_model.SGDClassifier (Logistic Regression behavior)
print("Training Sklearn SGDClassifier...")
sgd_sk = SGDClassifier(loss='log_loss', max_iter=100, tol=1e-3)
sgd_sk.fit(X_train_sk, y_train_sk)
sgd_acc = accuracy_score(y_test_sk, sgd_sk.predict(X_test_sk))

# Final Comparison Output
print("\n" + "="*30)
print(f"Scratch Model Accuracy: {accuracy * 100:.2f}%") # Using 'accuracy' from Cell 10
print(f"Sklearn LogReg Accuracy: {lr_acc * 100:.2f}%")
print(f"Sklearn SGD Accuracy:    {sgd_acc * 100:.2f}%")
print("="*30)

# Cleanup
del X_train_sk, X_test_sk
gc.collect()

Vectorizing data for Scikit-learn...
Training Sklearn LogisticRegression...


c:\Users\LENOVO\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


Training Sklearn SGDClassifier...

Scratch Model Accuracy: 8665.33%
Sklearn LogReg Accuracy: 88.93%
Sklearn SGD Accuracy:    88.43%


26